---
# PHẦN 1: SETUP TỰ ĐỘNG
---

## ⚙️ 1.1 — Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f"\n🔥 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🔥 VRAM: {vram:.1f} GB")
    if vram < 14:
        print("⚠️ VRAM < 14GB — Stage 2 có thể OOM. Nên dùng T4/V100/A100.")

## 📁 1.2 — Mount Drive + Cấu hình

Upload dữ liệu lên Drive theo cấu trúc:
```
MyDrive/TryHairStyle/processed_001/
MyDrive/TryHairStyle/processed_002/
MyDrive/TryHairStyle/eval_test/
  original/
  reference/
  manifest.json
```
Checkpoint và model export sẽ được lưu theo luồng hệ thống lên Hugging Face; Drive chỉ dùng cho dataset và eval set.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

GITHUB_REPO = 'https://github.com/Phatjhhoq8/TryHairstyle_Diffusion.git'
GITHUB_BRANCH = 'master'
DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
DRIVE_EVAL = f'{DRIVE_DATA}/eval_test'
HF_REPO_ID = 'halogenbr/tryhairstyle'

os.makedirs(DRIVE_DATA, exist_ok=True)
print(f'✅ Drive data: {DRIVE_DATA}')
print(f'✅ Drive eval: {DRIVE_EVAL}')
print(f'✅ HF repo: {HF_REPO_ID}')


## 📥 1.3 — Clone code từ GitHub

In [ ]:
import os

LOCAL_PROJECT = '/content/TryHairStyle'

if os.path.exists(LOCAL_PROJECT):
    print('⏭️ Project đã có. Pull latest...')
    !cd {LOCAL_PROJECT} && git pull origin {GITHUB_BRANCH}
else:
    print(f'📥 Cloning {GITHUB_REPO}...')
    !git clone -b {GITHUB_BRANCH} --depth 1 {GITHUB_REPO} {LOCAL_PROJECT}

assert os.path.exists(f'{LOCAL_PROJECT}/backend/training/train_stage2.py'), '❌ Clone failed'
print(f'\n✅ Project ready: {LOCAL_PROJECT}')


## 🤗 1.4 — Tải SDXL Inpainting Model (tự động)

Tải `diffusers/stable-diffusion-xl-1.0-inpainting-0.1` từ HuggingFace.
Tải thẳng vào local mỗi lần Colab khởi động (~10 phút).

In [ ]:
import os

LOCAL_PROJECT = '/content/TryHairStyle'
LOCAL_SDXL = f'{LOCAL_PROJECT}/backend/models/stable-diffusion/sd_xl_inpainting'

required_dirs = ['vae', 'unet', 'tokenizer', 'tokenizer_2', 'text_encoder', 'text_encoder_2', 'scheduler']

def check_sdxl(path):
    return os.path.exists(path) and all(os.path.exists(f'{path}/{d}') for d in required_dirs)

if check_sdxl(LOCAL_SDXL):
    print('✅ SDXL model đã có trên local')
else:
    print('📥 Đang tải SDXL Inpainting model từ HuggingFace -> local...')
    print('   (~10 phút, tải lại mỗi lần Colab reset)')
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    os.makedirs(LOCAL_SDXL, exist_ok=True)
    snapshot_download(
        repo_id='diffusers/stable-diffusion-xl-1.0-inpainting-0.1',
        local_dir=LOCAL_SDXL,
        ignore_patterns=['*.md', '*.png', '.gitattributes'],
    )
    print(f'✅ SDXL model đã tải về: {LOCAL_SDXL}')

assert check_sdxl(LOCAL_SDXL), f'❌ SDXL model incomplete at {LOCAL_SDXL}'
print('✅ SDXL model ready')


## 🔗 1.5 — Symlink Dataset Chunks + Eval Set


In [ ]:
import os, glob, shutil

LOCAL_PROJECT = '/content/TryHairStyle'
LOCAL_TRAINING = f'{LOCAL_PROJECT}/backend/training'
DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
DRIVE_EVAL = f'{DRIVE_DATA}/eval_test'

def _link(src, dst):
    if os.path.islink(dst):
        os.remove(dst)
    elif os.path.isdir(dst):
        shutil.rmtree(dst)
    elif os.path.exists(dst):
        os.remove(dst)
    os.symlink(src, dst)

chunks = sorted(glob.glob(f'{DRIVE_DATA}/processed_*'))
print(f'📂 Found {len(chunks)} chunk(s) trên Drive')
for c in chunks:
    name = os.path.basename(c)
    _link(c, f'{LOCAL_TRAINING}/{name}')
    meta = os.path.join(c, 'metadata.jsonl')
    n = sum(1 for l in open(meta)) if os.path.exists(meta) else '?'
    print(f'  🔗 {name}: {n} samples')

if not chunks:
    print('⚠️ KHÔNG TÌM THẤY DATASET!')
    print(f'   Upload các thư mục processed_NNN/ vào: {DRIVE_DATA}')

eval_local = f'{LOCAL_TRAINING}/eval_set'
if not os.path.exists(f'{DRIVE_EVAL}/manifest.json'):
    raise FileNotFoundError(f'Missing eval manifest: {DRIVE_EVAL}/manifest.json')
_link(DRIVE_EVAL, eval_local)
print(f'🔗 Eval set → Drive: {DRIVE_EVAL}')

os.makedirs(f'{LOCAL_TRAINING}/results', exist_ok=True)
print('✅ Symlinks ready!')


## 📦 1.6 — Cài Dependencies

In [ ]:
!pip install -q \
    diffusers>=0.36.0 \
    transformers>=4.36.0 \
    accelerate>=1.0.0 \
    safetensors>=0.7.0 \
    peft>=0.6.0 \
    bitsandbytes>=0.43.0 \
    insightface>=0.7.3 \
    facenet-pytorch>=2.6.0 \
    lpips>=0.1.4 \
    opencv-python-headless>=4.9.0 \
    scikit-image>=0.26.0 \
    easydict>=1.0 \
    deep-translator>=1.11.0

try:
    import xformers
    print(f"✅ xformers: {xformers.__version__}")
except ImportError:
    !pip install -q xformers

print("✅ Dependencies OK")

## 🔍 1.7 — Verify

In [ ]:
import json, os, glob

LOCAL_PROJECT = '/content/TryHairStyle'
LOCAL_TRAINING = f'{LOCAL_PROJECT}/backend/training'
errors = []

print('=' * 60)
print('🔍 VERIFY')
print('=' * 60)

for s in ['train_stage2.py', 'models/texture_encoder.py', 'export_model.py', 'evaluate_checkpoints.py', 'pregenerate_eval_outputs.py']:
    ok = os.path.exists(f'{LOCAL_TRAINING}/{s}')
    print(f"{'✅' if ok else '❌'} {s}")
    if not ok:
        errors.append(s)

sdxl = f'{LOCAL_PROJECT}/backend/models/stable-diffusion/sd_xl_inpainting'
for d in ['vae', 'unet', 'text_encoder', 'text_encoder_2']:
    ok = os.path.exists(f'{sdxl}/{d}')
    print(f"{'✅' if ok else '❌'} SDXL/{d}")
    if not ok:
        errors.append(f'SDXL/{d}')

chunks = sorted(glob.glob(f'{LOCAL_TRAINING}/processed_*'))
if not chunks:
    p = f'{LOCAL_TRAINING}/processed'
    if os.path.exists(p):
        chunks = [p]

total = 0
for c in chunks:
    meta = os.path.join(c, 'metadata.jsonl')
    if os.path.exists(meta):
        n = sum(1 for l in open(meta) if l.strip())
        total += n

manifest_path = f'{LOCAL_TRAINING}/eval_set/manifest.json'
if os.path.exists(manifest_path):
    manifest = json.load(open(manifest_path, 'r', encoding='utf-8'))
    samples = manifest.get('samples', [])
    print(f'✅ eval_set/manifest.json ({len(samples)} samples)')
    print(f'✅ eval_set/original: {len(glob.glob(f"{LOCAL_TRAINING}/eval_set/original/*"))} files')
    print(f'✅ eval_set/reference: {len(glob.glob(f"{LOCAL_TRAINING}/eval_set/reference/*"))} files')
else:
    print('❌ eval_set/manifest.json')
    errors.append('eval_set/manifest.json')

print('
' + '=' * 60)
if errors:
    print(f'⚠️ Missing: {errors}')
else:
    print(f'✅ ALL OK — {len(chunks)} chunk(s), {total:,} samples, eval set ready')
    print('🚀 Ready!')


## ⏱️ 1.8 — Anti-timeout

In [ ]:
import IPython
display(IPython.display.HTML('<script>setInterval(()=>{document.querySelectorAll("colab-connect-button").forEach(b=>b.click())},60000)</script>'))
print("✅ Auto-reconnect enabled")

---
# PHẦN 5: MONITORING
---

## 📊 Loss Chart

In [ ]:
from IPython.display import Image, display
import os

charts = '/tmp/training_checkpoints/charts'
latest = f'{charts}/loss_chart_latest.png'
if os.path.exists(latest):
    display(Image(filename=latest, width=900))
else:
    print("⚠️ Chưa có chart. Chạy training trước.")


## 📈 Training History

In [ ]:
import json, os

hp = '/tmp/training_checkpoints/training_history.json'
if os.path.exists(hp):
    h = json.load(open(hp))
    print(f"Steps: {len(h.get('total_loss',[]))} | Epochs: {len(h.get('epoch_avg_loss',[]))}")
    if h.get('epoch_avg_loss'):
        print(f"\n{'Ep':<5} {'Train':<12} {'Val':<12} {'LPIPS':<10} {'ID':<10} {'BG(dB)':<10}")
        print('-'*72)
        for i, t in enumerate(h['epoch_avg_loss']):
            v = h['val_loss'][i] if i < len(h.get('val_loss',[])) else 'N/A'
            l = h['val_lpips'][i] if i < len(h.get('val_lpips',[])) else 0
            fid = h['val_identity'][i] if i < len(h.get('val_identity',[])) else 0
            bg = h['val_background'][i] if i < len(h.get('val_background',[])) else 0
            print(f"{i+1:<5} {t:<12.6f} {(f'{v:.6f}' if isinstance(v,float) else v):<12} {(f'{l:.4f}' if l>0 else 'N/A'):<10} {(f'{fid:.4f}' if fid>0 else 'N/A'):<10} {(f'{bg:.2f}' if bg>0 else 'N/A'):<10}")
else:
    print('⚠️ Chưa có history.')


## 💾 Checkpoints on Hugging Face


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import list_repo_files

repo_id = 'halogenbr/tryhairstyle'
token = userdata.get('HUGFACE_TOKEN')
files = []
try:
    files = sorted([f for f in list_repo_files(repo_id=repo_id, repo_type='dataset', token=token) if f.startswith('checkpoints/')])
except Exception as e:
    print(f'⚠️ Không đọc được HF repo: {e}')

if files:
    print(f'☁️ HF repo: {repo_id}')
    for f in files:
        print(f'  {f}')
    epoch_files = [f for f in files if '/lora_epoch_' in f or '/injector_epoch_' in f]
    print(f'\nEpoch snapshots đang giữ: {len(epoch_files)} file')
else:
    print('⚠️ No checkpoints found on Hugging Face yet.')


## 🧹 Cleanup VRAM

In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🧹 Allocated={torch.cuda.memory_allocated()/1024**3:.2f}GB, Reserved={torch.cuda.memory_reserved()/1024**3:.2f}GB")
print("✅ Done")

---
# PHẦN 6: QUICK RESUME & TEST
---

## 🔄 QUICK RESUME (1 cell — sau disconnect)

Chạy **cell này duy nhất** khi Colab reconnect. Tự động:
1. Mount Drive
2. Clone/pull code
3. Tải/link SDXL model
4. Symlinks
5. Dependencies
6. Resume training

In [ ]:
# RESUME CONFIG
RESUME_STAGE = 2
RESUME_EPOCHS = 5
RESUME_BATCH = 4
RESUME_MAX_SAMPLES_PER_CHUNK = 0
RESUME_ACCUMULATION = 2
RESUME_RESOLUTION = 512
KEEP_LAST_N_EPOCHS = 5
RUN_BENCHMARK_AFTER_TRAIN = True

import json, os, sys, shutil, glob
from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_REPO = 'https://github.com/Phatjhhoq8/TryHairstyle_Diffusion.git'
GITHUB_BRANCH = 'master'
DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
DRIVE_EVAL = f'{DRIVE_DATA}/eval_test'
LP = '/content/TryHairStyle'
LT = f'{LP}/backend/training'

if os.path.exists(LP):
    !cd {LP} && git pull origin {GITHUB_BRANCH} 2>/dev/null || true
else:
    !git clone -b {GITHUB_BRANCH} --depth 1 {GITHUB_REPO} {LP}

sdxl_local = f'{LP}/backend/models/stable-diffusion/sd_xl_inpainting'
req = ['vae','unet','text_encoder','text_encoder_2','tokenizer','tokenizer_2','scheduler']
if not (os.path.exists(sdxl_local) and all(os.path.exists(f'{sdxl_local}/{d}') for d in req)):
    print('📥 Downloading SDXL model to local...')
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    os.makedirs(sdxl_local, exist_ok=True)
    snapshot_download('diffusers/stable-diffusion-xl-1.0-inpainting-0.1', local_dir=sdxl_local, ignore_patterns=['*.md','*.png','.gitattributes'])
print('✅ SDXL ready')

def _lnk(src, dst):
    if os.path.islink(dst):
        os.remove(dst)
    elif os.path.isdir(dst):
        shutil.rmtree(dst)
    elif os.path.exists(dst):
        os.remove(dst)
    os.symlink(src, dst)

for c in sorted(glob.glob(f'{DRIVE_DATA}/processed_*')):
    _lnk(c, f'{LT}/{os.path.basename(c)}')
if not os.path.exists(f'{DRIVE_EVAL}/manifest.json'):
    raise FileNotFoundError(f'Missing eval manifest: {DRIVE_EVAL}/manifest.json')
_lnk(DRIVE_EVAL, f'{LT}/eval_set')

!pip install -q diffusers>=0.36.0 transformers>=4.36.0 accelerate>=1.0.0 safetensors>=0.7.0 peft>=0.6.0 bitsandbytes>=0.43.0 insightface>=0.7.3 facenet-pytorch>=2.6.0 lpips>=0.1.4 opencv-python-headless>=4.9.0 scikit-image>=0.26.0 easydict>=1.0 deep-translator>=1.11.0 2>/dev/null

os.chdir(LP)
sys.path.insert(0, LP)
os.environ['COLAB_GPU'] = '1'
os.environ['HUGFACE_TOKEN'] = userdata.get('HUGFACE_TOKEN')
os.environ['HF_REPO_ID'] = 'halogenbr/tryhairstyle'
print(f"☁️ HF checkpoint repo: {os.environ['HF_REPO_ID']}")

print(f'\n🔄 Resuming Stage {RESUME_STAGE}...')
print('=' * 60)

if RESUME_STAGE == 1:
    !python -u backend/training/models/texture_encoder.py --epochs {RESUME_EPOCHS} --batch-size 16 --resume
else:
    !python -u backend/training/train_stage2.py --epochs {RESUME_EPOCHS} --batch-size {RESUME_BATCH} --max-samples-per-chunk {RESUME_MAX_SAMPLES_PER_CHUNK} --resolution {RESUME_RESOLUTION} --accumulation {RESUME_ACCUMULATION} --keep-last-n-epochs {KEEP_LAST_N_EPOCHS}

print('\n' + '=' * 60 + '\n📦 STAGE 3: Export\n' + '=' * 60)
!python -u backend/training/export_model.py

if RUN_BENCHMARK_AFTER_TRAIN:
    print('\n' + '=' * 60 + '\n🧪 STAGE 4: Eval Generation\n' + '=' * 60)
    !python -u backend/training/pregenerate_eval_outputs.py --checkpoint backend/models/deep_hair_v1.safetensors --eval_set_dir backend/training/eval_set --output_dir backend/training/results/deep_hair_v1/generated --overwrite
    print('\n' + '=' * 60 + '\n📊 STAGE 5: Benchmark\n' + '=' * 60)
    !python -u backend/training/evaluate_checkpoints.py --checkpoints_dir backend/models --eval_set_dir backend/training/eval_set --results_dir backend/training/results --device cuda

    print('\n' + '=' * 60 + '\n☁️ STAGE 6: Upload benchmark JSON to HF\n' + '=' * 60)
    from huggingface_hub import upload_file
    for local_path, remote_name in [
        ('backend/training/results/leaderboard.json', 'eval_results/leaderboard.json'),
        ('backend/training/results/deep_hair_v1/summary.json', 'eval_results/deep_hair_v1_summary.json'),
        ('backend/training/results/deep_hair_v1/generation_summary.json', 'eval_results/deep_hair_v1_generation_summary.json'),
    ]:
        if os.path.exists(local_path):
            upload_file(path_or_fileobj=local_path, path_in_repo=remote_name, repo_id=os.environ['HF_REPO_ID'], repo_type='dataset', token=os.environ['HUGFACE_TOKEN'], commit_message=f'benchmark: {os.path.basename(local_path)}')
            print(f'  ☁️ Uploaded {remote_name}')

    summary_path = 'backend/training/results/deep_hair_v1/summary.json'
    if os.path.exists(summary_path):
        summary = json.load(open(summary_path, 'r', encoding='utf-8'))
        print('\n✅ Benchmark summary:')
        print(json.dumps(summary, indent=2, ensure_ascii=False))


## 🧪 Smoke Test (test nhanh)

In [ ]:
import os, sys
LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'
from google.colab import userdata
os.environ['HUGFACE_TOKEN'] = userdata.get('HUGFACE_TOKEN')
os.environ['HF_REPO_ID'] = 'halogenbr/tryhairstyle'

print("🧪 TEST: Stage 1 (50 samples, 2ep)")
# !python -u backend/training/models/texture_encoder.py --epochs 2 --batch-size 8 --max-samples 50

print("
🧪 TEST: Stage 2 (50 samples/chunk, 1ep)")
!python -u backend/training/train_stage2.py --epochs 1 --batch-size 2 --max-samples-per-chunk 4 --accumulation 4 --fresh --chunk-names "processed_001" --save-steps 2 --keep-last-n-epochs 2

print("
🧪 TEST: Stage 3")
!python -u backend/training/export_model.py

print("
✅ SMOKE TEST OK")


## 🚀 Full Pipeline (Stage 1→2→3)

In [ ]:
import json, os, sys, time
from google.colab import userdata

LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'
os.environ['HUGFACE_TOKEN'] = userdata.get('HUGFACE_TOKEN')
os.environ['HF_REPO_ID'] = 'halogenbr/tryhairstyle'

FP_S1_EPOCHS = 20
FP_S2_EPOCHS = 1
FP_S2_BATCH = 2
FP_S2_MAX_SAMPLES = 0
FP_S2_ACCUMULATION = 4
FP_KEEP_LAST_N_EPOCHS = 5
FP_RUN_BENCHMARK = True

t0 = time.time()

print('\n' + '='*60 + '\n🧬 STAGE 1\n' + '='*60)
t1 = time.time()
# !python -u backend/training/models/texture_encoder.py --epochs {FP_S1_EPOCHS} --batch-size 16 --resume
s1 = (time.time()-t1)/60

print('\n' + '='*60 + '\n🎨 STAGE 2\n' + '='*60)
t2 = time.time()
!python -u backend/training/train_stage2.py --epochs {FP_S2_EPOCHS} --batch-size {FP_S2_BATCH} --max-samples-per-chunk {FP_S2_MAX_SAMPLES} --accumulation {FP_S2_ACCUMULATION} --chunk-names "processed_001" --keep-last-n-epochs {FP_KEEP_LAST_N_EPOCHS}
s2 = (time.time()-t2)/60

print('\n' + '='*60 + '\n📦 STAGE 3\n' + '='*60)
!python -u backend/training/export_model.py

if FP_RUN_BENCHMARK:
    print('\n' + '='*60 + '\n🧪 STAGE 4\n' + '='*60)
    !python -u backend/training/pregenerate_eval_outputs.py --checkpoint backend/models/deep_hair_v1.safetensors --eval_set_dir backend/training/eval_set --output_dir backend/training/results/deep_hair_v1/generated --overwrite
    print('\n' + '='*60 + '\n📊 STAGE 5\n' + '='*60)
    !python -u backend/training/evaluate_checkpoints.py --checkpoints_dir backend/models --eval_set_dir backend/training/eval_set --results_dir backend/training/results --device cuda

tt = (time.time()-t0)/60
print(f"\n{'='*60}\n✅ DONE! S1={s1:.0f}min, S2={s2:.0f}min, Total={tt:.0f}min ({tt/60:.1f}h)")

summary_path = 'backend/training/results/deep_hair_v1/summary.json'
if os.path.exists(summary_path):
    summary = json.load(open(summary_path, 'r', encoding='utf-8'))
    print('\nLatest benchmark summary:')
    print(json.dumps(summary, indent=2, ensure_ascii=False))


## 📊 Benchmark Summary


In [ ]:
import json, os

leaderboard_path = '/content/TryHairStyle/backend/training/results/leaderboard.json'
summary_path = '/content/TryHairStyle/backend/training/results/deep_hair_v1/summary.json'

if os.path.exists(leaderboard_path):
    leaderboard = json.load(open(leaderboard_path, 'r', encoding='utf-8'))
    print('🏆 Leaderboard:')
    print(json.dumps(leaderboard, indent=2, ensure_ascii=False))
else:
    print('⚠️ Chưa có leaderboard.json')

if os.path.exists(summary_path):
    summary = json.load(open(summary_path, 'r', encoding='utf-8'))
    print('\n📌 Summary:')
    print(json.dumps(summary, indent=2, ensure_ascii=False))
else:
    print('⚠️ Chưa có summary.json')


In [ ]:
# ═══════════════════════════════════════════════════════
# 🔍 CHI TIẾT CẤU TRÚC THƯ MỤC DATASET TRÊN DRIVE
# ═══════════════════════════════════════════════════════
import os, glob

DRIVE_DATA = '/content/drive/MyDrive/TryHairStyle'
MAX_FILES_PREVIEW = 5  # Số file mẫu hiển thị cho mỗi thư mục (đặt 0 = hiển thị TẤT CẢ)

chunks = sorted(glob.glob(f'{DRIVE_DATA}/processed_*'))

if not chunks:
    print(f"⚠️ Không tìm thấy thư mục processed_* nào trong: {DRIVE_DATA}")
else:
    total_chunks = len(chunks)
    total_all_files = 0

    for chunk_path in chunks:
        chunk_name = os.path.basename(chunk_path)
        print(f"\n{'═'*70}")
        print(f"📂 {chunk_name}/")
        print(f"{'═'*70}")

        # Liệt kê tất cả items trong chunk (thư mục + file ở root)
        items = sorted(os.listdir(chunk_path))
        dirs_in_chunk = []
        files_in_root = []

        for item in items:
            full = os.path.join(chunk_path, item)
            if os.path.isdir(full):
                dirs_in_chunk.append(item)
            elif os.path.isfile(full):
                files_in_root.append(item)

        # Hiển thị files ở root (metadata.jsonl, etc.)
        if files_in_root:
            print(f"\n  📄 Files ở root ({len(files_in_root)} file):")
            for f in files_in_root:
                size = os.path.getsize(os.path.join(chunk_path, f))
                if size > 1024*1024:
                    size_str = f"{size/1024/1024:.1f} MB"
                elif size > 1024:
                    size_str = f"{size/1024:.1f} KB"
                else:
                    size_str = f"{size} B"
                print(f"     • {f}  ({size_str})")
                total_all_files += 1

        # Hiển thị từng thư mục con
        if dirs_in_chunk:
            for subdir in dirs_in_chunk:
                subdir_path = os.path.join(chunk_path, subdir)
                files = sorted([
                    f for f in os.listdir(subdir_path)
                    if os.path.isfile(os.path.join(subdir_path, f))
                ])
                n = len(files)
                total_all_files += n

                print(f"\n  📁 {subdir}/ — {n} file(s)")

                if n == 0:
                    print(f"     (trống)")
                elif MAX_FILES_PREVIEW == 0 or n <= MAX_FILES_PREVIEW:
                    # Hiển thị tất cả
                    for f in files:
                        print(f"     • {f}")
                else:
                    # Hiển thị mẫu đầu + cuối
                    for f in files[:MAX_FILES_PREVIEW]:
                        print(f"     • {f}")
                    if n > MAX_FILES_PREVIEW:
                        print(f"     ... và {n - MAX_FILES_PREVIEW} file khác")
        else:
            print(f"  (không có thư mục con)")

    # Tổng kết
    print(f"\n{'═'*70}")
    print(f"📊 TỔNG KẾT")
    print(f"{'═'*70}")
    print(f"  📂 Số chunks: {total_chunks}")
    print(f"  📁 Tổng files: {total_all_files:,}")


## export nhanh

In [ ]:
import os, sys
LOCAL_PROJECT = '/content/TryHairStyle'
os.chdir(LOCAL_PROJECT)
sys.path.insert(0, LOCAL_PROJECT)
os.environ['COLAB_GPU'] = '1'
from google.colab import userdata
os.environ['HUGFACE_TOKEN'] = userdata.get('HUGFACE_TOKEN')
os.environ['HF_REPO_ID'] = 'halogenbr/tryhairstyle'
print(f"HF repo: {os.environ['HF_REPO_ID']}")
!python -m backend.training.export_compare